# Scikit-learn for Machine Learning Students

A practical, student-friendly notebook covering the most useful **scikit-learn** tools and workflows for ML.

## Learning path
1. What scikit-learn is
2. Datasets and train/test split
3. Preprocessing and scaling
4. Encoding categorical data
5. Missing-value handling
6. Feature selection and dimensionality reduction
7. Regression
8. Classification
9. Clustering
10. Model evaluation
11. Cross-validation
12. Hyperparameter tuning
13. Pipelines
14. Feature engineering
15. Saving/loading models
16. Complete ML workflows

> **Core ML workflow:** Data → Split → Preprocess → Train → Predict → Evaluate → Tune → Save


In [ ]:
# Install if needed:
# !pip install scikit-learn pandas numpy matplotlib seaborn joblib

import numpy as np
import pandas as pd

import sklearn
print("scikit-learn version:", sklearn.__version__)


## 1. Built-in Datasets

Scikit-learn provides small datasets that are excellent for learning and experimentation.

In [ ]:
from sklearn.datasets import (
    load_iris,
    load_diabetes,
    load_breast_cancer,
    load_digits,
    make_classification,
    make_regression,
    make_blobs
)

# Iris classification dataset
iris = load_iris()

print("Features shape:", iris.data.shape)
print("Target shape:", iris.target.shape)
print("Feature names:", iris.feature_names)
print("Target names:", iris.target_names)


In [ ]:
# Convert a sklearn dataset into a DataFrame
iris_df = pd.DataFrame(
    iris.data,
    columns=iris.feature_names
)

iris_df["target"] = iris.target

display(iris_df.head())


## 2. Train/Test Split

The most basic ML evaluation idea:

- Training data → model learns from it
- Test data → evaluate on unseen data

`random_state` makes the split reproducible.

In [ ]:
from sklearn.model_selection import train_test_split

X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)


## 3. Feature Scaling

Many algorithms work better when numerical features are on comparable scales.

Common scalers:
- `StandardScaler` → mean 0, standard deviation 1
- `MinMaxScaler` → usually scales to 0–1
- `RobustScaler` → less sensitive to outliers

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training mean:", X_train_scaled.mean(axis=0))
print("Training std:", X_train_scaled.std(axis=0))


### Important: `fit_transform()` vs `transform()`

Use:

```python
scaler.fit_transform(X_train)
```

for training data.

Use:

```python
scaler.transform(X_test)
```

for test data.

This prevents **data leakage** from the test set into preprocessing.

In [ ]:
# Correct pattern
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 4. Min-Max Scaling

In [ ]:
from sklearn.preprocessing import MinMaxScaler

minmax = MinMaxScaler()

X_train_minmax = minmax.fit_transform(X_train)
X_test_minmax = minmax.transform(X_test)

print("Minimum:", X_train_minmax.min(axis=0))
print("Maximum:", X_train_minmax.max(axis=0))


## 5. Robust Scaling

In [ ]:
from sklearn.preprocessing import RobustScaler

robust = RobustScaler()

X_train_robust = robust.fit_transform(X_train)
X_test_robust = robust.transform(X_test)


## 6. Missing Values

`SimpleImputer` can replace missing values using strategies such as:
- mean
- median
- most frequent
- constant

In [ ]:
from sklearn.impute import SimpleImputer

data = pd.DataFrame({
    "Age": [20, 25, np.nan, 35, 40],
    "Salary": [40000, np.nan, 60000, 70000, 80000]
})

imputer = SimpleImputer(strategy="mean")

data_imputed = imputer.fit_transform(data)

print(data_imputed)


In [ ]:
# Median is often useful when outliers exist
median_imputer = SimpleImputer(strategy="median")

data_median = median_imputer.fit_transform(data)

print(data_median)


## 7. Categorical Encoding

Machine-learning models usually require numerical input.

`OneHotEncoder` converts categories into binary columns.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

cat_data = pd.DataFrame({
    "Department": ["IT", "HR", "Finance", "IT", "HR"]
})

encoder = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore"
)

encoded = encoder.fit_transform(cat_data)

print(encoded)
print("Categories:", encoder.categories_)


## 8. Label Encoding

`LabelEncoder` is commonly used for a **target label**, not as a general replacement for one-hot encoding of input features.

In [ ]:
from sklearn.preprocessing import LabelEncoder

labels = ["Spam", "Ham", "Spam", "Ham", "Spam"]

label_encoder = LabelEncoder()

encoded_labels = label_encoder.fit_transform(labels)

print("Encoded:", encoded_labels)
print("Classes:", label_encoder.classes_)

# Convert back
print("Decoded:", label_encoder.inverse_transform(encoded_labels))


## 9. ColumnTransformer

Different columns often need different preprocessing.

Example:
- numerical columns → StandardScaler
- categorical columns → OneHotEncoder

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

df = pd.DataFrame({
    "Age": [20, 30, 40, 25],
    "Salary": [40000, 60000, 90000, 50000],
    "Department": ["IT", "HR", "IT", "Finance"]
})

X = df[["Age", "Salary", "Department"]]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), ["Age", "Salary"]),
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["Department"])
    ]
)

X_processed = preprocessor.fit_transform(X)

print(X_processed)


## 10. Regression: Linear Regression

Linear regression predicts a continuous numerical value.

Formula:

`ŷ = Xw + b`

In [ ]:
from sklearn.linear_model import LinearRegression

X, y = make_regression(
    n_samples=200,
    n_features=3,
    noise=20,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

model = LinearRegression()

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Coefficients:", model.coef_)
print("Intercept:", model.intercept_)
print("Predictions:", y_pred[:5])


## 11. Regression Metrics

Common regression metrics:
- MAE → average absolute error
- MSE → average squared error
- RMSE → square root of MSE
- R² → proportion of variance explained by the model

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R²  :", r2)


## 12. Ridge Regression

Ridge adds **L2 regularization** to reduce overly large coefficients and help control overfitting.

In [ ]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0)

ridge.fit(X_train, y_train)

ridge_pred = ridge.predict(X_test)

print("R²:", r2_score(y_test, ridge_pred))


## 13. Lasso Regression

Lasso uses **L1 regularization** and can shrink some coefficients to exactly zero, which can help with feature selection.

In [ ]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.1)

lasso.fit(X_train, y_train)

lasso_pred = lasso.predict(X_test)

print("R²:", r2_score(y_test, lasso_pred))
print("Coefficients:", lasso.coef_)


## 14. Logistic Regression

Despite its name, logistic regression is commonly used for **classification**.

For binary classification it models a probability using the sigmoid function.

In [ ]:
from sklearn.linear_model import LogisticRegression

X, y = load_breast_cancer(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_model = LogisticRegression(max_iter=5000)

log_model.fit(X_train_scaled, y_train)

y_pred = log_model.predict(X_test_scaled)
y_prob = log_model.predict_proba(X_test_scaled)

print("Predictions:", y_pred[:10])
print("Probabilities:", y_prob[:5])


## 15. Classification Metrics

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1       :", f1_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


## 16. Confusion Matrix Explained

For binary classification:

- True Positive (TP)
- True Negative (TN)
- False Positive (FP)
- False Negative (FN)

From these we derive accuracy, precision, recall, and F1.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

tn, fp, fn, tp = cm.ravel()

print("TN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)


## 17. ROC-AUC

ROC-AUC evaluates ranking quality using predicted probabilities.

For binary classification, use the probability of the positive class.

In [ ]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(y_test, y_prob[:, 1])

print("ROC-AUC:", auc)


## 18. Decision Tree Classifier

Decision trees split data using feature-based rules.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

tree.fit(X_train, y_train)

tree_pred = tree.predict(X_test)

print("Accuracy:", accuracy_score(y_test, tree_pred))


## 19. Random Forest

Random Forest combines many decision trees to form an ensemble model.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, rf_pred))


## 20. Feature Importance

Tree-based models can estimate the importance of input features.

In [ ]:
importance = pd.Series(
    rf.feature_importances_,
    index=load_breast_cancer().feature_names
).sort_values(ascending=False)

display(importance.head(10))


## 21. K-Nearest Neighbors (KNN)

KNN predicts based on nearby observations.

Because distance matters, scaling is usually important.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)

knn.fit(X_train_scaled, y_train)

knn_pred = knn.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, knn_pred))


## 22. Support Vector Machine (SVM)

In [ ]:
from sklearn.svm import SVC

svm = SVC(
    kernel="rbf",
    probability=True,
    random_state=42
)

svm.fit(X_train_scaled, y_train)

svm_pred = svm.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, svm_pred))


## 23. Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB()

nb.fit(X_train, y_train)

nb_pred = nb.predict(X_test)

print("Accuracy:", accuracy_score(y_test, nb_pred))


## 24. Gradient Boosting

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gb.fit(X_train, y_train)

gb_pred = gb.predict(X_test)

print("Accuracy:", accuracy_score(y_test, gb_pred))


## 25. Clustering with K-Means

K-Means is an **unsupervised learning** algorithm.

It groups observations into `k` clusters.

In [ ]:
from sklearn.cluster import KMeans

X_blobs, _ = make_blobs(
    n_samples=300,
    centers=3,
    cluster_std=1.0,
    random_state=42
)

kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

clusters = kmeans.fit_predict(X_blobs)

print("First 20 cluster labels:", clusters[:20])
print("Cluster centers:\n", kmeans.cluster_centers_)


## 26. K-Means Inertia

Inertia is the sum of squared distances of samples to their closest cluster center.

It is often examined with the **elbow method**.

In [ ]:
inertias = []

for k in range(1, 10):
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    model.fit(X_blobs)
    inertias.append(model.inertia_)

print("Inertias:", inertias)


## 27. Hierarchical / Agglomerative Clustering

In [ ]:
from sklearn.cluster import AgglomerativeClustering

agg = AgglomerativeClustering(
    n_clusters=3
)

agg_labels = agg.fit_predict(X_blobs)

print(agg_labels[:20])


## 28. Dimensionality Reduction with PCA

PCA reduces the number of features while trying to retain important variance.

In [ ]:
from sklearn.decomposition import PCA

X, y = load_iris(return_X_y=True)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2)

X_pca = pca.fit_transform(X_scaled)

print("Original shape:", X.shape)
print("PCA shape:", X_pca.shape)
print("Explained variance ratio:", pca.explained_variance_ratio_)
print("Total explained variance:", pca.explained_variance_ratio_.sum())


## 29. Cross-Validation

Instead of relying on one train/test split, cross-validation evaluates a model across multiple folds.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

X, y = load_breast_cancer(return_X_y=True)

model = LogisticRegression(max_iter=5000)

scores = cross_val_score(
    model,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

print("Fold scores:", scores)
print("Mean CV accuracy:", scores.mean())


## 30. K-Fold Cross-Validation

In [ ]:
from sklearn.model_selection import KFold, cross_val_score

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

model = LogisticRegression(max_iter=5000)

scores = cross_val_score(
    model,
    X,
    y,
    cv=kf,
    scoring="accuracy"
)

print("Scores:", scores)
print("Mean:", scores.mean())


## 31. Stratified K-Fold

For classification, `StratifiedKFold` attempts to preserve class proportions in each fold.

In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    LogisticRegression(max_iter=5000),
    X,
    y,
    cv=skf,
    scoring="accuracy"
)

print("Scores:", scores)
print("Mean:", scores.mean())


## 32. Hyperparameter Tuning with GridSearchCV

Grid search tests specified combinations of hyperparameters.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)

param_grid = {
    "n_estimators": [50, 100],
    "max_depth": [3, 5, None],
    "min_samples_split": [2, 5]
}

grid = GridSearchCV(
    rf,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X, y)

print("Best parameters:", grid.best_params_)
print("Best CV score:", grid.best_score_)


## 33. RandomizedSearchCV

Instead of testing every combination, randomized search samples a specified number of combinations.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

rf = RandomForestClassifier(random_state=42)

param_distributions = {
    "n_estimators": [50, 100, 200],
    "max_depth": [3, 5, 10, None],
    "min_samples_split": [2, 5, 10]
}

random_search = RandomizedSearchCV(
    rf,
    param_distributions=param_distributions,
    n_iter=10,
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)

random_search.fit(X, y)

print("Best parameters:", random_search.best_params_)
print("Best score:", random_search.best_score_)


## 34. Pipelines

A pipeline combines preprocessing and modeling into one reproducible workflow.

This is especially useful for avoiding preprocessing mistakes and data leakage.

In [ ]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

pipeline.fit(X_train, y_train)

pred = pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))


## 35. Pipeline + GridSearchCV

In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000))
])

param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100]
}

grid = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy"
)

grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best CV score:", grid.best_score_)


## 36. Feature Selection with SelectKBest

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(
    score_func=f_classif,
    k=5
)

X_selected = selector.fit_transform(X_train, y_train)

print("Original features:", X_train.shape[1])
print("Selected features:", X_selected.shape[1])


## 37. Recursive Feature Elimination (RFE)

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

estimator = LogisticRegression(max_iter=5000)

rfe = RFE(
    estimator=estimator,
    n_features_to_select=5
)

rfe.fit(X_train_scaled, y_train)

print("Selected:", rfe.support_)
print("Ranking:", rfe.ranking_)


## 38. Polynomial Features

Polynomial features can allow a linear model to capture nonlinear relationships.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

X_small = np.array([[1], [2], [3], [4]])

poly = PolynomialFeatures(
    degree=2,
    include_bias=False
)

X_poly = poly.fit_transform(X_small)

print(X_poly)
print("Feature names:", poly.get_feature_names_out(["X"]))


## 39. Learning Curves

Learning curves help study how model performance changes as the amount of training data increases.

In [ ]:
from sklearn.model_selection import learning_curve
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=5000)

train_sizes, train_scores, validation_scores = learning_curve(
    model,
    X,
    y,
    cv=5,
    scoring="accuracy",
    train_sizes=np.linspace(0.1, 1.0, 5)
)

print("Train sizes:", train_sizes)
print("Mean training scores:", train_scores.mean(axis=1))
print("Mean validation scores:", validation_scores.mean(axis=1))


## 40. Prediction Probabilities

In [ ]:
model = LogisticRegression(max_iter=5000)

model.fit(X_train_scaled, y_train)

probabilities = model.predict_proba(X_test_scaled)

print("First 5 probability rows:")
print(probabilities[:5])


## 41. Precision-Recall Curve

In [ ]:
from sklearn.metrics import precision_recall_curve

y_scores = model.predict_proba(X_test_scaled)[:, 1]

precision, recall, thresholds = precision_recall_curve(
    y_test,
    y_scores
)

print("Precision:", precision[:5])
print("Recall:", recall[:5])


## 42. Regression Cross-Validation

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor

X_reg, y_reg = make_regression(
    n_samples=300,
    n_features=5,
    noise=15,
    random_state=42
)

reg_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

scores = cross_val_score(
    reg_model,
    X_reg,
    y_reg,
    cv=5,
    scoring="r2"
)

print("R² scores:", scores)
print("Mean R²:", scores.mean())


## 43. Model Persistence

Save a trained model so it can be reused without retraining.

`joblib` is commonly used for scikit-learn objects.

In [ ]:
import joblib

# Example:
model = LogisticRegression(max_iter=5000)
model.fit(X_train_scaled, y_train)

joblib.dump(model, "logistic_model.pkl")
joblib.dump(scaler, "scaler.pkl")

print("Models saved.")


In [ ]:
# Load the saved model and scaler
loaded_model = joblib.load("logistic_model.pkl")
loaded_scaler = joblib.load("scaler.pkl")

new_data_scaled = loaded_scaler.transform(X_test[:3])
print("Loaded-model predictions:", loaded_model.predict(new_data_scaled))


## 44. Complete ML Pipeline Example

This example demonstrates a professional-style workflow:

**Data → Split → Pipeline → Cross-validation → Tuning → Test evaluation**

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

X, y = load_breast_cancer(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000))
])

param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100]
}

search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

search.fit(X_train, y_train)

final_pred = search.predict(X_test)

print("Best parameters:", search.best_params_)
print("Best CV score:", search.best_score_)
print("Test accuracy:", accuracy_score(y_test, final_pred))
print(classification_report(y_test, final_pred))


# 45. Algorithm Selection Cheat Sheet

| Problem | Algorithms to learn |
|---|---|
| Binary classification | Logistic Regression, KNN, SVM, Decision Tree, Random Forest |
| Multi-class classification | Logistic Regression, KNN, SVM, Trees, Random Forest, Gradient Boosting |
| Regression | Linear Regression, Ridge, Lasso, Decision Tree, Random Forest, Gradient Boosting |
| Clustering | K-Means, Agglomerative Clustering |
| Dimensionality reduction | PCA |
| Feature selection | SelectKBest, RFE |
| Nonlinear relationships | Trees, Random Forest, Gradient Boosting, SVM |
| High-dimensional data | Regularization, feature selection, PCA |


# 46. ML Workflow Cheat Sheet

```text
                 RAW DATA
                     ↓
             EDA / DATA CHECK
                     ↓
             Train / Test Split
                     ↓
          ┌──────────┴──────────┐
          ↓                     ↓
     Numerical             Categorical
     Imputation             Encoding
     Scaling
          └──────────┬──────────┘
                     ↓
                 Pipeline
                     ↓
              Model Training
                     ↓
              Cross-Validation
                     ↓
             Hyperparameter Tuning
                     ↓
                  Predict
                     ↓
                Evaluation
                     ↓
               Save Model
                     ↓
               Deployment
```

## Most important scikit-learn concepts for ML students

1. `train_test_split()`
2. `StandardScaler()`
3. `SimpleImputer()`
4. `OneHotEncoder()`
5. `ColumnTransformer()`
6. `Pipeline()`
7. `LinearRegression()`
8. `LogisticRegression()`
9. `DecisionTreeClassifier()`
10. `RandomForestClassifier()`
11. `KNeighborsClassifier()`
12. `SVC()`
13. `GradientBoostingClassifier()`
14. `KMeans()`
15. `PCA()`
16. `cross_val_score()`
17. `GridSearchCV()`
18. `RandomizedSearchCV()`
19. `classification_report()`
20. `confusion_matrix()`
21. `accuracy_score()`
22. `precision_score()`
23. `recall_score()`
24. `f1_score()`
25. `roc_auc_score()`

> **Key teaching point:** Students should understand the workflow and why each step is used, not just memorize algorithms.
